# 03 Official Baseline Run

Run the official qualitative baseline on the locked 20-clip subset using `Grounding DINO + SAM2` with no re-grounding.

- Uses the same bootstrap path as D1: `bash setup_colab.sh --with-models`
- Uses the existing locked `subset_manifest.csv`; it does not rebuild the manifest
- Writes per-clip outputs, an aggregate baseline summary, a reviewed baseline table, and 5 success + 5 failure sample exports


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
REPO_URL = 'https://github.com/ChiCoTheLaAnh/ProjectFinalCS415.git'
REPO_DIR = '/content/ProjectFinalCS415'

!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"


In [ ]:
%cd /content/ProjectFinalCS415
!bash setup_colab.sh --with-models


In [ ]:
import shutil
import subprocess
import sys
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Torch CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())

nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi is not None:
    gpu_name = subprocess.run(
        [nvidia_smi, '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    print('GPU:', gpu_name)

if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required for the official baseline run.')


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/cv-final-project')
BASELINE_NAME = 'baseline_groundingdino_sam2_no_regrounding'
MANIFEST_PATH = Path('/content/ProjectFinalCS415/data/processed/subset_manifest.csv')
OUTPUT_ROOT = DRIVE_ROOT / 'results' / 'quantitative' / BASELINE_NAME
SAMPLES_ROOT = DRIVE_ROOT / 'results' / 'final_figures' / 'baseline_samples'
GROUNDING_CKPT = DRIVE_ROOT / 'checkpoints' / 'groundingdino_swint_ogc.pth'
SAM2_CKPT = DRIVE_ROOT / 'checkpoints' / 'sam2.1_hiera_small.pt'
SUBSET_LIMIT = None
DEVICE = 'cuda'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SAMPLES_ROOT.mkdir(parents=True, exist_ok=True)
print('Manifest path:', MANIFEST_PATH)
print('Output root:', OUTPUT_ROOT)
print('Samples root:', SAMPLES_ROOT)


In [ ]:
for required_path in (MANIFEST_PATH, GROUNDING_CKPT, SAM2_CKPT):
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required path: {required_path}')
    print(required_path)


In [ ]:
import pandas as pd

manifest_df = pd.read_csv(MANIFEST_PATH)
display(manifest_df.head(20))
print('Total rows:', len(manifest_df))
print('Selected rows:', int((manifest_df['selected'].astype(str) == '1').sum()))
print('Primary tag counts:')
print(manifest_df[manifest_df['selected'].astype(str) == '1']['primary_tag'].value_counts())


In [ ]:
selected_df = pd.read_csv(MANIFEST_PATH)
selected_df['selected'] = selected_df['selected'].astype(str)
chosen = selected_df[selected_df['selected'] == '1'].copy()
expected_selected = len(chosen)
if expected_selected < 1:
    raise ValueError('The official baseline manifest must contain at least one selected clip.')
invalid_tags = sorted(set(chosen['primary_tag']) - {'easy', 'occlusion', 'small_object', 'crowded'})
if invalid_tags:
    raise ValueError(f'Invalid primary_tag values: {invalid_tags}')
if (chosen['notes'].fillna('').str.strip() == '').any():
    raise ValueError('Every selected row must have a non-empty notes value because notes is used as the prompt.')
display(chosen[['clip_id', 'video_path', 'primary_tag', 'notes']])
duplicate_clip_ids = chosen[chosen['clip_id'].duplicated(keep=False)]['clip_id'].tolist()
if duplicate_clip_ids:
    print('Duplicate clip_id values kept in the current baseline seed:', sorted(set(duplicate_clip_ids)))
print(chosen['primary_tag'].value_counts())


In [ ]:
%cd /content/ProjectFinalCS415
import subprocess

cmd = [
    'python3', 'scripts/run_eval_subset.py',
    '--config', 'configs/base.yaml',
    '--manifest', str(MANIFEST_PATH),
    '--output_dir', str(OUTPUT_ROOT),
    '--grounding_ckpt', str(GROUNDING_CKPT),
    '--sam2_ckpt', str(SAM2_CKPT),
    '--device', DEVICE,
]
if SUBSET_LIMIT is not None:
    cmd.extend(['--limit', str(SUBSET_LIMIT)])
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json

summary_path = OUTPUT_ROOT / 'subset_run_summary.json'
if not summary_path.exists():
    raise FileNotFoundError(f'Missing subset run summary: {summary_path}')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
if summary.get('num_completed') != expected_selected:
    raise ValueError(f'Expected {expected_selected} completed clips, found {summary.get("num_completed")}')
for clip in summary.get('clips', []):
    if clip.get('artifacts', {}).get('video_mode') != 'sam2_video_predictor':
        raise ValueError(f"Clip {clip.get('clip_id')} did not stay on sam2_video_predictor")
print(json.dumps(summary, indent=2))
print('Tag counts:', summary.get('tag_counts', {}))
print('Completed clips:', summary.get('num_completed'))


In [ ]:
%cd /content/ProjectFinalCS415
import subprocess

cmd = [
    'python3', 'scripts/export_results.py',
    '--input-dir', str(OUTPUT_ROOT),
    '--samples-dir', str(SAMPLES_ROOT),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## Manual Review Step

Open `baseline_table.csv` under the baseline output root and review all 20 rows.

Allowed `review_label` values:

- `good_tracking`
- `partial_tracking`
- `drift`
- `wrong_object`
- `no_detection`
- `fallback`

Requirements before the final export cell:

- every row has `review_label`
- every row has a one-line `review_note`
- at least 5 rows are `good_tracking`
- at least 5 rows are failure examples, using `partial_tracking` only if needed to fill the last slots


In [ ]:
baseline_table_path = OUTPUT_ROOT / 'baseline_table.csv'
if not baseline_table_path.exists():
    raise FileNotFoundError(f'Missing baseline table: {baseline_table_path}')
baseline_df = pd.read_csv(baseline_table_path)
display(baseline_df)
print('Review label counts:')
print(baseline_df['review_label'].fillna('').value_counts())


In [ ]:
%cd /content/ProjectFinalCS415
import subprocess

cmd = [
    'python3', 'scripts/export_results.py',
    '--input-dir', str(OUTPUT_ROOT),
    '--samples-dir', str(SAMPLES_ROOT),
    '--require-reviewed',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
baseline_summary_path = OUTPUT_ROOT / 'baseline_summary.json'
baseline_examples_path = OUTPUT_ROOT / 'baseline_examples.csv'
if not baseline_summary_path.exists():
    raise FileNotFoundError(f'Missing baseline summary: {baseline_summary_path}')
if not baseline_examples_path.exists():
    raise FileNotFoundError(f'Missing baseline examples: {baseline_examples_path}')

baseline_summary = json.loads(baseline_summary_path.read_text(encoding='utf-8'))
examples_df = pd.read_csv(baseline_examples_path)
display(examples_df)
if len(examples_df) != 10:
    raise ValueError(f'Expected 10 baseline examples, found {len(examples_df)}')
bucket_counts = examples_df['sample_bucket'].value_counts().to_dict()
if bucket_counts.get('success', 0) != 5:
    raise ValueError(f'Expected 5 success examples, found {bucket_counts.get("success", 0)}')
if bucket_counts.get('failure', 0) + bucket_counts.get('failure_borderline', 0) != 5:
    raise ValueError('Expected 5 failure examples in total.')
print(json.dumps(baseline_summary, indent=2))
print('Example bucket counts:', bucket_counts)
print('Samples root:', SAMPLES_ROOT)
